# Once the models are trained check how do they react when the env observation is rotated mirrored and changed. How do they react to color change?

In [1]:
import sys
import os
sys.path.append(os.path.abspath("/data/I6347325/work_space/STORM"))

In [2]:
import os
import gymnasium
import minigrid
from collections import deque
from functools import partial
from tqdm import tqdm
import numpy as np
import cv2
import torch
from einops import rearrange
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import colorama
from sub_models.constants import DEVICE
import glob

from sub_models.world_models import WorldModel
import sub_models.agents as agents
from utils import seed_np_torch, load_config
from train import build_world_model, build_agent
from env_wrapper import LifeLossInfo
import train

/home/I6347325/miniconda3/envs/env_RL/lib/python3.13/site-packages/torchrl/data/replay_buffers/samplers.py:34: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)


In [3]:
class RunParams:
    def __init__(self, env_names, run_name: str, config_path):
        self.run_name = run_name
        self.seed = 1
        self.config_path = config_path
        # self.trajectory_path = f"D_TRAJ/{self._env_name}.pkl"
        self.env_names = env_names

        self.conf = load_config(self.config_path)
        self.print_args()

    def print_args(self):
        print(colorama.Fore.GREEN + "Arguments:" + colorama.Style.RESET_ALL)
        print(colorama.Fore.GREEN + "-----------------" + colorama.Style.RESET_ALL)
        print(
            colorama.Fore.GREEN
            + "run_name: "
            + colorama.Style.RESET_ALL
            + self.run_name
        )
        print(
            colorama.Fore.GREEN + "seed: " + colorama.Style.RESET_ALL + str(self.seed)
        )
        # print(colorama.Fore.GREEN + "config_path: " + colorama.Style.RESET_ALL + self.config_path)
        print(colorama.Fore.GREEN + "env_name: " + colorama.Style.RESET_ALL)
        print(self.env_names)
        print(colorama.Fore.GREEN + "-----------------" + colorama.Style.RESET_ALL)


def process_visualize(img):
    img = img.astype("uint8")
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.resize(img, (320, 320))
    return img


def build_single_env(env_name: str, image_size: int, env_observablity: str = "Full"):
    """
    Build a single env with wrappers and preprocesses env.
    """
    env = gymnasium.make(env_name, render_mode="rgb_array")
    # Convert int to tuple as gymnasium.wrappers.ResizeObservation requires tuple
    if isinstance(image_size, int):
        image_size = (image_size, image_size)
    if env_observablity == "Full":
        env = minigrid.wrappers.RGBImgObsWrapper(env)
    elif env_observablity == "Partial":
        env = minigrid.wrappers.RGBImgPartialObsWrapper(env)
    else:
        raise ValueError(f"Unknown env observability {env_observablity}")
    env = minigrid.wrappers.ImgObsWrapper(env)  # Sets obs = obs["rgb"], discards others
    env = gymnasium.wrappers.ResizeObservation(env, shape=image_size)
    # env = env_wrapper.LifeLossInfo(env)

    return env


def build_vec_env(env_names: list[str], image_size: int, env_observablity):
    """
    Build a vectorized env with n=num_envs parallel envs.
    """
    env_fns = [
        partial(build_single_env, env_name, image_size, env_observablity)
        for env_name in env_names
    ]
    vec_env = gymnasium.vector.AsyncVectorEnv(env_fns=env_fns)
    return vec_env



In [4]:
def inspect_model(
    env_names: list[str],
    max_steps: int,
    env_observablity: str,
    image_size: int,
    world_model: WorldModel,
    agent: agents.ActorCriticAgent,
    imagine_batch_length: int = 32,
    save_obs: bool = False,
    env_names_map: dict = None,
    run_name: str = "default_run",
):
    world_model.eval()
    agent.eval()
    vec_env = build_vec_env(env_names, image_size, env_observablity)

    num_envs = len(env_names)
    sum_reward = np.zeros(num_envs)
    current_obs, _ = vec_env.reset()
    context_obs = deque(maxlen=imagine_batch_length)
    context_action = deque(maxlen=imagine_batch_length)

    episode_counts =0
    final_rewards = []
    hidden_feats_env = []
    embeddings, labels = [], []
    while episode_counts<= max_steps:
        with torch.no_grad():
            if len(context_action) == 0:
                action = vec_env.action_space.sample()
            else:
                context_latent = world_model.encode_obs(
                    torch.cat(list(context_obs), dim=1)
                )
                model_context_action = np.stack(list(context_action), axis=1)
                model_context_action = torch.Tensor(model_context_action).to(DEVICE)
                prior_flattened_sample, last_dist_feat = (
                    world_model.calc_last_dist_feat(
                        context_latent, model_context_action
                    )
                )
                action = agent.sample_as_env_action(
                    torch.cat([prior_flattened_sample, last_dist_feat], dim=-1),
                    greedy=False,
                )

        context_obs.append(
            rearrange(torch.Tensor(current_obs).to(DEVICE), "B H W C -> B 1 C H W")
            / 255
        )
        context_action.append(action)
        # take a step in the environment
        obs, reward, done, truncated, info = vec_env.step(action)
        if save_obs:
            for i in range(num_envs):
                if env_names_map is not None:
                    env_name = env_names_map[env_names[i]]
                else:
                    env_name = env_names[i]
                env_folder = f"./frames/{run_name}/{env_name}"
                os.makedirs(env_folder, exist_ok=True)
                frame = process_visualize(obs[i])
                frame_path = os.path.join(env_folder, f"step_{episode_counts}.png")
                cv2.imwrite(frame_path, frame)

        # check if the episode is done
        done_flag = np.logical_or(done, truncated)
        if done_flag.any():
            for i in range(num_envs):
                if done_flag[i]:
                    final_rewards.append(sum_reward[i])
                    # sum_reward[i] = 0
        # update current_obs, current_info and sum_reward
        sum_reward += reward
        current_obs = obs
        episode_counts += 1

    return final_rewards, sum_reward


def run_env_variations(run_params, ckpt_root, model_step, max_episode=16, save_obs=False, env_names_map=None, exp_name=None): 
    # set seed
    seed_np_torch(seed=0)

    # build and load model/agent
    dummy_env = build_single_env(
        run_params.env_names[0], run_params.conf.BasicSettings.ImageSize
    )
    action_dim = dummy_env.action_space.n
    world_model = build_world_model(run_params.conf, action_dim)
    agent = build_agent(run_params.conf, action_dim)
    
    world_model.load_state_dict(
        torch.load(f"{ckpt_root}/world_model_{model_step}.pth", map_location=DEVICE)
    )
    agent.load_state_dict(
        torch.load(f"{ckpt_root}/agent_{model_step}.pth", map_location=DEVICE)
    )
    final_rewards, sum_reward = inspect_model(
        env_names=run_params.env_names,
        max_steps=max_episode,
        env_observablity=run_params.conf.BasicSettings.EnvObservability,
        image_size=run_params.conf.BasicSettings.ImageSize,
        world_model=world_model,
        agent=agent,
        imagine_batch_length=run_params.conf.JointTrainAgent.ImagineBatchLength,
        save_obs= save_obs,
        run_name=exp_name if exp_name is not None else run_params.run_name,
        env_names_map=env_names_map,
    )
    return final_rewards, sum_reward 

# Empty with different orientation

In [5]:
env_names = [
        "MiniGrid-Empty-8x8-v0",
        "MiniGrid-Empty-8x8-CRG-O1",
        "MiniGrid-Empty-8x8-CRG-O2",
        "MiniGrid-Empty-8x8-CRG-O3",
        
    ]
env_names_map = {
        "MiniGrid-Empty-8x8-v0": "Empty-original",
        "MiniGrid-Empty-8x8-CRG-O1": "Empty-CRG-O1",
        "MiniGrid-Empty-8x8-CRG-O2": "Empty-CRG-O2",
        "MiniGrid-Empty-8x8-CRG-O3": "Empty-CRG-O3",
        }
seed = 0


In [ ]:
## Full Observation Baseline
RUN_NAME = "MultiEnvFullObs-Baseline_v1" #"EmptyFullObs-Baseline_v1" # "MemoryFullObs-Baseline_v1" #
ckpt_root = f"../ckpt/{RUN_NAME}/"
config_path = f"../config_files/STORM.yaml"
STEP = 70000
run_params = RunParams(env_names, RUN_NAME, config_path)
exp_name = "MapInversion-Empty-FullObs-Baseline"
final_rewards, sum_reward  = run_env_variations(run_params, ckpt_root, STEP, 32, True, env_names_map, exp_name)

In [7]:
final_rewards, sum_reward 

([np.float64(0.0), np.float64(0.961328125)],
 array([1.92265625, 0.        , 0.        , 0.        ]))

In [ ]:
## Part Observation Baseline
RUN_NAME = "MultiEnvPartObs-Baseline_v1" #"EmptyFullObs-Baseline_v1" # "MemoryFullObs-Baseline_v1" #
ckpt_root = f"../ckpt/{RUN_NAME}/"
config_path = f"../config_files/STORM.yaml"
STEP = 70000
run_params = RunParams(env_names, RUN_NAME, config_path)
exp_name = "MapInversion-Empty-PartObs-Baseline"
final_rewards, sum_reward  = run_env_variations(run_params, ckpt_root, STEP, 16, True, env_names_map, exp_name)

Arguments:
-----------------
run_name: MultiEnvPartObs-Baseline_v1
seed: 1
env_name: 
['MiniGrid-Empty-8x8-v0', 'MiniGrid-Empty-8x8-CRG-O1', 'MiniGrid-Empty-8x8-CRG-O2', 'MiniGrid-Empty-8x8-CRG-O3']
-----------------


In [10]:
final_rewards, sum_reward

([np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)],
 array([0.9578125 , 0.95078125, 0.95078125, 0.94726562]))

In [6]:
## Full Observation TEM
RUN_NAME = "MultiEnv-FullObs-TEM-v1" 
ckpt_root = f"../ckpt/{RUN_NAME}/"
config_path = f"../config_files/STORM.yaml"
STEP = 70000
run_params = RunParams(env_names, RUN_NAME, config_path)
exp_name = "MapInversion-Empty-FullObs-TEM"
final_rewards, sum_reward  = run_env_variations(run_params, ckpt_root, STEP, 16, True, env_names_map, exp_name)
final_rewards, sum_reward

Arguments:
-----------------
run_name: MultiEnv-FullObs-TEM-v1
seed: 1
env_name: 
['MiniGrid-Empty-8x8-v0', 'MiniGrid-Empty-8x8-CRG-O1', 'MiniGrid-Empty-8x8-CRG-O2', 'MiniGrid-Empty-8x8-CRG-O3']
-----------------


([np.float64(0.0)], array([0.9578125, 0.       , 0.       , 0.       ]))

In [8]:
## Partial Observation TEM
RUN_NAME = "MultiEnv-PartObs-TEM-v1"
ckpt_root = f"../ckpt/{RUN_NAME}/"
config_path = f"../config_files/STORM.yaml"
STEP = 70000
run_params = RunParams(env_names, RUN_NAME, config_path)
exp_name = "MapInversion-Empty-PartObs-TEM"
final_rewards, sum_reward  = run_env_variations(run_params, ckpt_root, STEP, 16, True, env_names_map, exp_name)
final_rewards, sum_reward

Arguments:
-----------------
run_name: MultiEnv-PartObs-TEM-v1
seed: 1
env_name: 
['MiniGrid-Empty-8x8-v0', 'MiniGrid-Empty-8x8-CRG-O1', 'MiniGrid-Empty-8x8-CRG-O2', 'MiniGrid-Empty-8x8-CRG-O3']
-----------------


([np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)],
 array([0.95078125, 0.95078125, 0.94726562, 0.95429688]))

In [52]:
def calculate_plot_tsne(embeddings, labels):
    # ---- t-SNE ----
    embeddings = np.array(embeddings)
    tsne = TSNE(n_components=2, perplexity = 2, learning_rate=200) #perplexity=8,
    emb_2d = tsne.fit_transform(embeddings)

    # ---- Plot ----
    label_map = {label: idx for idx, label in enumerate(set(labels))}
    colors = [label_map[l] for l in labels]
    plt.figure(figsize=(10, 8))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, cmap="tab10")
    for i, label in enumerate(labels):
        plt.annotate(label, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=8, alpha=0.7)
    plt.title(
        f"t-SNE of World Model Transformer Embeddings ({'Mean' if USE_MEAN_POOL else 'Last'} Token)"
    )
    plt.show()


In [6]:
## Full Observation Baseline v2
RUN_NAME = "MultiEnvFullObs-Baseline_v1" #"EmptyFullObs-Baseline_v1" # "MemoryFullObs-Baseline_v1" #
ckpt_root = f"../ckpt/{RUN_NAME}/"
config_path = f"../config_files/STORM.yaml"
STEP = 5000
run_params = RunParams(env_names, RUN_NAME, config_path)
exp_name = "MapInversion-Empty-FullObs-Baseline-step5k"
final_rewards, sum_reward  = run_env_variations(run_params, ckpt_root, STEP, 16, True, env_names_map, exp_name)
final_rewards, sum_reward

Arguments:
-----------------
run_name: MultiEnvFullObs-Baseline_v1
seed: 1
env_name: 
['MiniGrid-Empty-8x8-v0', 'MiniGrid-Empty-8x8-CRG-O1', 'MiniGrid-Empty-8x8-CRG-O2', 'MiniGrid-Empty-8x8-CRG-O3']
-----------------


([np.float64(0.0)], array([0.95429688, 0.        , 0.        , 0.        ]))